# Example 1: Tube MPC vs Regular MPC

This notebook compares a **regular MPC** controller and a **Tube MPC** controller on the same disturbed double-integrator model used in `ps11`:

$$
x(k+1)=Ax(k)+Bu(k)+B_d w(k),
$$

with

$$
A=\begin{bmatrix}1 & 1\\0 & 1\end{bmatrix},\qquad
B=\begin{bmatrix}0\\1\end{bmatrix},\qquad
B_d=\begin{bmatrix}0\\1\end{bmatrix}.
$$

The disturbance is modeled as a constant offset plus optional bounded random noise.

- **Regular MPC** plans as if there is no disturbance.
- **Tube MPC** plans a nominal trajectory inside tightened constraints and applies an ancillary feedback law around it.

In [ ]:
# Optional dependency install (uncomment if needed)
# %pip install numpy scipy matplotlib pandas ipywidgets cvxpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cvxpy as cp

from matplotlib.patches import Rectangle
from scipy.linalg import solve_discrete_are
from IPython.display import display

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (14, 9)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

A = np.array([[1.0, 1.0], [0.0, 1.0]])
B = np.array([[0.0], [1.0]])
Bd = np.array([[0.0], [1.0]])

x_min = np.array([-5.0, -5.0])
x_max = np.array([5.0, 5.0])
u_min = -0.8
u_max = 0.8

Q_mpc = np.diag([1.2, 0.35])
R_mpc = np.array([[0.18]])
P_mpc = solve_discrete_are(A, B, Q_mpc, R_mpc)

Q_lqr = np.diag([3.0, 1.2])
R_lqr = np.array([[0.7]])
P_lqr = solve_discrete_are(A, B, Q_lqr, R_lqr)
K_tube = -np.linalg.solve(R_lqr + B.T @ P_lqr @ B, B.T @ P_lqr @ A)
Acl = A + B @ K_tube

mpc_problem_cache = {}
tube_design_cache = {}


## Interactive Comparison

Use the controls to change the initial state, horizon, disturbance assumptions, and noise level.

- The **design disturbance bound** is the uncertainty size Tube MPC prepares for.
- The **actual constant disturbance** is what the simulated plant actually sees.
- The **noise amplitude** adds extra random bounded perturbations on top of that offset.

Reading the plots:

- Top left: regular MPC phase-plane rollout.
- Top right: Tube MPC phase-plane rollout.
- Bottom left: position trajectory $x_1(k)$.
- Bottom right: input trajectory $u(k)$.

For Tube MPC, the dashed box is the tightened constraint set used by the nominal optimizer, and the light rectangles visualize the local tube around the nominal path.

In [ ]:
def _round_key(arr, digits=6):
    arr = np.array(arr, dtype=float).reshape(-1)
    return tuple(np.round(arr, digits))


def get_mpc_problem(horizon, x_lo, x_hi, u_lo, u_hi):
    key = (
        int(horizon),
        _round_key(x_lo),
        _round_key(x_hi),
        round(float(u_lo), 6),
        round(float(u_hi), 6),
    )
    if key in mpc_problem_cache:
        bundle = mpc_problem_cache[key]
        if all(name in bundle for name in ["problem", "x0", "x_ref", "X", "U"]):
            return bundle
        del mpc_problem_cache[key]

    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))
    x0_param = cp.Parameter(2)
    x_ref_param = cp.Parameter(2)

    constraints = [
        X[:, 0] == x0_param,
        X[:, horizon] >= x_lo,
        X[:, horizon] <= x_hi,
    ]
    cost = 0

    for k in range(horizon):
        cost += cp.quad_form(X[:, k] - x_ref_param, Q_mpc) + cp.quad_form(U[:, k], R_mpc)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k],
            X[:, k] >= x_lo,
            X[:, k] <= x_hi,
            U[:, k] >= u_lo,
            U[:, k] <= u_hi,
        ]

    cost += cp.quad_form(X[:, horizon] - x_ref_param, P_mpc)
    problem = cp.Problem(cp.Minimize(cost), constraints)
    bundle = {"problem": problem, "x0": x0_param, "x_ref": x_ref_param, "X": X, "U": U}
    mpc_problem_cache[key] = bundle
    return bundle


def solve_mpc_step(x0, horizon, x_lo, x_hi, u_lo, u_hi, x_ref=None, solver=cp.OSQP):
    bundle = get_mpc_problem(horizon, x_lo, x_hi, u_lo, u_hi)
    bundle["x0"].value = np.array(x0, dtype=float)
    if x_ref is None:
        x_ref = np.zeros(2)
    bundle["x_ref"].value = np.array(x_ref, dtype=float)
    bundle["problem"].solve(
        solver=solver,
        warm_start=True,
        verbose=False,
        eps_abs=1e-5,
        eps_rel=1e-5,
        max_iter=20000,
    )
    if bundle["U"].value is None:
        return None, bundle["problem"].status, None, None
    return (
        float(bundle["U"].value[0, 0]),
        bundle["problem"].status,
        np.array(bundle["X"].value, dtype=float),
        np.array(bundle["U"].value, dtype=float),
    )


def compute_tube_design(disturbance_bound, n_terms=80):
    w_bound = max(float(disturbance_bound), 0.0)
    key = round(w_bound, 6)
    if key in tube_design_cache:
        return tube_design_cache[key]

    series = np.zeros(2)
    power = np.eye(2)
    for _ in range(n_terms):
        series += np.abs((power @ Bd).reshape(-1))
        power = Acl @ power

    eps_x = w_bound * series
    eps_u = float(np.sum(np.abs(K_tube).reshape(-1) * eps_x))
    x_lo_tight = x_min + eps_x
    x_hi_tight = x_max - eps_x
    u_lo_tight = u_min + eps_u
    u_hi_tight = u_max - eps_u
    feasible = bool(np.all(x_lo_tight < x_hi_tight) and (u_lo_tight < u_hi_tight))

    design = {
        "w_bound": w_bound,
        "eps_x": eps_x,
        "eps_u": eps_u,
        "x_lo_tight": x_lo_tight,
        "x_hi_tight": x_hi_tight,
        "u_lo_tight": u_lo_tight,
        "u_hi_tight": u_hi_tight,
        "feasible": feasible,
    }
    tube_design_cache[key] = design
    return design


def sample_disturbance_sequence(sim_steps, d_const, noise_amp, seed):
    rng = np.random.default_rng(int(seed))
    disturbances = []
    for _ in range(sim_steps):
        noise = rng.uniform(-noise_amp, noise_amp) if noise_amp > 0 else 0.0
        disturbances.append(float(d_const + noise))
    return np.array(disturbances, dtype=float)


def simulate_regular_mpc(x0, x_ref, horizon, sim_steps, disturbances):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    predicted_paths = []
    nominal_path = [x.copy()]
    statuses = []
    violated = False

    for k in range(sim_steps):
        u_cmd, status, X_pred, _ = solve_mpc_step(x, horizon, x_min, x_max, u_min, u_max, x_ref=x_ref)
        statuses.append(status)
        predicted_paths.append(X_pred.copy() if X_pred is not None else None)
        if u_cmd is None:
            break

        x_next = A @ x + B[:, 0] * u_cmd + Bd[:, 0] * disturbances[k]
        nominal_next = X_pred[:, 1] if X_pred is not None and X_pred.shape[1] > 1 else (A @ x + B[:, 0] * u_cmd)
        violated = violated or bool(np.any(x_next < x_min - 1e-8) or np.any(x_next > x_max + 1e-8) or u_cmd < u_min - 1e-8 or u_cmd > u_max + 1e-8)

        xs.append(x_next.copy())
        us.append(float(u_cmd))
        nominal_path.append(nominal_next.copy())
        x = x_next

    return {
        "x": np.array(xs),
        "u": np.array(us),
        "nominal": np.array(nominal_path),
        "predicted_paths": predicted_paths,
        "statuses": statuses,
        "violated": violated,
    }


def simulate_tube_mpc(x0, x_ref, horizon, sim_steps, disturbances, tube_design):
    x = np.array(x0, dtype=float).copy()
    x_nom = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    x_nom_hist = [x_nom.copy()]
    us = []
    u_nom_hist = []
    u_fb_hist = []
    predicted_paths = []
    statuses = []
    violated = False
    tube_respected = True
    input_clip_count = 0
    eps_x = tube_design["eps_x"]

    for k in range(sim_steps):
        u_nom, status, X_pred, _ = solve_mpc_step(
            x_nom,
            horizon,
            tube_design["x_lo_tight"],
            tube_design["x_hi_tight"],
            tube_design["u_lo_tight"],
            tube_design["u_hi_tight"],
            x_ref=x_ref,
        )
        statuses.append(status)
        predicted_paths.append(X_pred.copy() if X_pred is not None else None)
        if u_nom is None:
            break

        e = x - x_nom
        u_fb = float((K_tube @ e).item())
        u_raw = float(u_nom + u_fb)
        u_cmd = float(np.clip(u_raw, u_min, u_max))
        if abs(u_cmd - u_raw) > 1e-8:
            input_clip_count += 1

        x_next = A @ x + B[:, 0] * u_cmd + Bd[:, 0] * disturbances[k]
        x_nom_next = A @ x_nom + B[:, 0] * float(u_nom)
        e_next = x_next - x_nom_next

        violated = violated or bool(np.any(x_next < x_min - 1e-8) or np.any(x_next > x_max + 1e-8) or u_cmd < u_min - 1e-8 or u_cmd > u_max + 1e-8)
        tube_respected = tube_respected and bool(np.all(np.abs(e) <= eps_x + 1e-8) and np.all(np.abs(e_next) <= eps_x + 1e-8))

        xs.append(x_next.copy())
        x_nom_hist.append(x_nom_next.copy())
        us.append(u_cmd)
        u_nom_hist.append(float(u_nom))
        u_fb_hist.append(u_fb)
        x = x_next
        x_nom = x_nom_next

    return {
        "x": np.array(xs),
        "x_nom": np.array(x_nom_hist),
        "u": np.array(us),
        "u_nom": np.array(u_nom_hist),
        "u_fb": np.array(u_fb_hist),
        "predicted_paths": predicted_paths,
        "statuses": statuses,
        "violated": violated,
        "tube_respected": tube_respected,
        "input_clip_count": input_clip_count,
    }


def add_constraint_box(ax, x_lo, x_hi, color, linestyle="-", linewidth=1.8, alpha=1.0):
    rect = Rectangle((x_lo[0], x_lo[1]), x_hi[0] - x_lo[0], x_hi[1] - x_lo[1], fill=False, ec=color, ls=linestyle, lw=linewidth, alpha=alpha)
    ax.add_patch(rect)


def plot_phase_panel(ax, actual, nominal, predicted_paths, title, target_point, tube_design=None, show_tube=True, show_plans=True):
    add_constraint_box(ax, x_min, x_max, color="black", linestyle="-", linewidth=1.8, alpha=0.95)

    if tube_design is not None:
        add_constraint_box(ax, tube_design["x_lo_tight"], tube_design["x_hi_tight"], color="#1565c0", linestyle="--", linewidth=1.5, alpha=0.9)

    if show_plans:
        for X_pred in predicted_paths:
            if X_pred is None:
                continue
            ax.plot(X_pred[0, :], X_pred[1, :], color="#90caf9", lw=0.9, alpha=0.28)

    if nominal is not None and len(nominal) > 0:
        ax.plot(nominal[:, 0], nominal[:, 1], color="#1e88e5", lw=1.8, ls="--", marker="o", ms=3.5, alpha=0.9, label="nominal")

    ax.plot(actual[:, 0], actual[:, 1], color="#111111", lw=2.2, marker="s", ms=4.0, alpha=0.95, label="actual")
    ax.scatter([actual[0, 0]], [actual[0, 1]], color="#2e7d32", s=70, zorder=6, label="start")
    ax.scatter([target_point[0]], [target_point[1]], marker="x", color="#c62828", s=90, linewidths=2.2, zorder=6, label="target")

    if tube_design is not None and show_tube and nominal is not None:
        eps_x = tube_design["eps_x"]
        for point in nominal:
            tube_patch = Rectangle(
                (point[0] - eps_x[0], point[1] - eps_x[1]),
                2.0 * eps_x[0],
                2.0 * eps_x[1],
                fill=True,
                fc="#64b5f6",
                ec="#42a5f5",
                lw=0.8,
                alpha=0.12,
            )
            ax.add_patch(tube_patch)

    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_xlim(-5.2, 5.2)
    ax.set_ylim(-5.2, 5.2)
    ax.grid(True, alpha=0.35)


def summarise_rollout(name, rollout, target_point, disturbances, tube_design=None):
    xs = rollout["x"]
    max_dev = float(np.max(np.linalg.norm(xs, axis=1))) if len(xs) else np.nan
    final_err = float(np.linalg.norm(xs[-1] - target_point)) if len(xs) else np.nan
    state_margin = float(np.min(np.hstack([xs - x_min, x_max - xs]))) if len(xs) else np.nan
    status_ok = all(status in ("optimal", "optimal_inaccurate") for status in rollout["statuses"])
    row = {
        "controller": name,
        "max ||x||": max_dev,
        "final target error": final_err,
        "min state margin": state_margin,
        "max |u|": float(np.max(np.abs(rollout["u"]))) if len(rollout["u"]) else np.nan,
        "constraint violation": bool(rollout["violated"]),
        "solver ok": status_ok,
    }
    if tube_design is not None:
        row["tube respected"] = bool(rollout["tube_respected"])
        row["input clips"] = int(rollout["input_clip_count"])
        row["design |w| bound"] = float(tube_design["w_bound"])
    return row


def render_tube_mpc_dashboard(
    x1_0,
    x2_0,
    x1_ref,
    horizon,
    sim_steps,
    design_bound,
    actual_disturbance,
    noise_amp,
    seed,
    show_tube,
    show_plans,
):
    x0 = np.array([x1_0, x2_0], dtype=float)
    x_ref = np.array([x1_ref, 0.0], dtype=float)
    disturbances = sample_disturbance_sequence(sim_steps, actual_disturbance, noise_amp, seed)
    tube_design = compute_tube_design(design_bound)

    if not tube_design["feasible"]:
        display(pd.DataFrame([
            {
                "message": "Tube tightening is infeasible for this disturbance bound.",
                "design disturbance bound": float(design_bound),
                "suggestion": "Reduce the design bound or widen constraints.",
            }
        ]))
        return

    regular = simulate_regular_mpc(x0, x_ref, horizon, sim_steps, disturbances)
    tube = simulate_tube_mpc(x0, x_ref, horizon, sim_steps, disturbances, tube_design)

    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 0.9])
    ax_phase_regular = fig.add_subplot(gs[0, 0])
    ax_phase_tube = fig.add_subplot(gs[0, 1], sharex=ax_phase_regular, sharey=ax_phase_regular)
    ax_x1 = fig.add_subplot(gs[1, 0])
    ax_u = fig.add_subplot(gs[1, 1])

    plot_phase_panel(
        ax_phase_regular,
        regular["x"],
        regular["nominal"],
        regular["predicted_paths"],
        "Regular MPC",
        x_ref,
        tube_design=None,
        show_tube=False,
        show_plans=show_plans,
    )
    plot_phase_panel(
        ax_phase_tube,
        tube["x"],
        tube["x_nom"],
        tube["predicted_paths"],
        "Tube MPC",
        x_ref,
        tube_design=tube_design,
        show_tube=show_tube,
        show_plans=show_plans,
    )

    k_reg = np.arange(len(regular["x"]))
    k_tube = np.arange(len(tube["x"]))
    ax_x1.plot(k_reg, regular["x"][:, 0], color="#424242", lw=2.0, marker="o", ms=3.5, label="Regular actual")
    ax_x1.plot(k_tube, tube["x"][:, 0], color="#1565c0", lw=2.0, marker="s", ms=3.5, label="Tube actual")
    ax_x1.plot(k_tube, tube["x_nom"][:, 0], color="#90caf9", lw=1.5, ls="--", label="Tube nominal")
    ax_x1.axhline(x_ref[0], color="#c62828", lw=1.2, ls="--", alpha=0.8, label="target x1")
    ax_x1.axhline(x_max[0], color="black", lw=1.0, ls=":")
    ax_x1.axhline(x_min[0], color="black", lw=1.0, ls=":")
    ax_x1.set_title("Position trajectory $x_1(k)$")
    ax_x1.set_xlabel("step $k$")
    ax_x1.set_ylabel("$x_1$")
    ax_x1.legend(loc="upper right")

    k_u_reg = np.arange(len(regular["u"]))
    k_u_tube = np.arange(len(tube["u"]))
    ax_u.step(k_u_reg, regular["u"], where="post", color="#424242", lw=2.0, label="Regular input")
    ax_u.step(k_u_tube, tube["u"], where="post", color="#1565c0", lw=2.0, label="Tube applied")
    if len(tube["u_nom"]) > 0:
        ax_u.step(k_u_tube, tube["u_nom"], where="post", color="#90caf9", lw=1.5, ls="--", label="Tube nominal input")
    ax_u.axhline(u_max, color="black", lw=1.0, ls=":")
    ax_u.axhline(u_min, color="black", lw=1.0, ls=":")
    ax_u.set_title("Input trajectory $u(k)$")
    ax_u.set_xlabel("step $k$")
    ax_u.set_ylabel("$u$")
    ax_u.legend(loc="upper right")

    phase_handles = [
        plt.Line2D([0], [0], color="black", lw=1.8, label="State constraints"),
        plt.Line2D([0], [0], color="#1565c0", lw=1.5, ls="--", label="Tightened nominal constraints"),
        plt.Line2D([0], [0], color="#111111", lw=2.2, label="Actual state trajectory"),
        plt.Line2D([0], [0], color="#1e88e5", lw=1.8, ls="--", label="Nominal trajectory"),
        plt.Line2D([0], [0], color="#64b5f6", lw=8, alpha=0.35, label="Tube cross-sections"),
    ]
    fig.legend(handles=phase_handles, loc="lower center", ncol=5, framealpha=0.95)
    fig.suptitle(
        f"Tube MPC comparison: target x1 = {x_ref[0]:.2f}, design |w| <= {design_bound:.2f}, actual offset = {actual_disturbance:.2f}, noise amplitude = {noise_amp:.2f}",
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.show()

    if regular["violated"] and not tube["violated"]:
        display(pd.DataFrame([{"message": "Clear robustness win: regular MPC violates constraints here, while Tube MPC stays inside the admissible set."}]))
    elif (not regular["violated"]) and (not tube["violated"]):
        display(pd.DataFrame([{"message": "Both controllers stay feasible here. Increase the target, disturbance, or shorten the horizon to stress regular MPC more."}]))
    else:
        display(pd.DataFrame([{"message": "This is a harsh case for both controllers. Reduce the actual disturbance or increase the Tube design bound."}]))

    if np.max(np.abs(disturbances)) > tube_design["w_bound"] + 1e-8:
        display(pd.DataFrame([{"warning": "Actual disturbance exceeds the Tube MPC design bound, so robustness is no longer guaranteed."}]))

    summary = pd.DataFrame([
        summarise_rollout("Regular MPC", regular, x_ref, disturbances),
        summarise_rollout("Tube MPC", tube, x_ref, disturbances, tube_design=tube_design),
    ])
    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(3)
    display(summary)

    diagnostics = pd.DataFrame([
        {
            "target state": f"[{x_ref[0]:.2f}, {x_ref[1]:.2f}]",
            "actual disturbance range": f"[{disturbances.min():.3f}, {disturbances.max():.3f}]",
            "tube epsilon x1": round(float(tube_design["eps_x"][0]), 3),
            "tube epsilon x2": round(float(tube_design["eps_x"][1]), 3),
            "tube epsilon u": round(float(tube_design["eps_u"]), 3),
            "tightened x1 interval": f"[{tube_design['x_lo_tight'][0]:.2f}, {tube_design['x_hi_tight'][0]:.2f}]",
            "tightened x2 interval": f"[{tube_design['x_lo_tight'][1]:.2f}, {tube_design['x_hi_tight'][1]:.2f}]",
        }
    ])
    display(diagnostics)


x1_init_sl = widgets.FloatSlider(value=0.0, min=-4.5, max=4.5, step=0.25, description="initial x1", continuous_update=False)
x2_init_sl = widgets.FloatSlider(value=0.0, min=-4.5, max=4.5, step=0.25, description="initial x2", continuous_update=False)
x1_ref_sl = widgets.FloatSlider(value=-4.0, min=-4.0, max=4.4, step=0.1, description="target x1", continuous_update=False)
horizon_sl = widgets.IntSlider(value=5, min=3, max=12, step=1, description="Horizon", continuous_update=False)
sim_steps_sl = widgets.IntSlider(value=16, min=6, max=28, step=1, description="Sim steps", continuous_update=False)
design_bound_sl = widgets.FloatSlider(value=0.18, min=0.0, max=0.25, step=0.01, description="design |w|", continuous_update=False)
actual_dist_sl = widgets.FloatSlider(value=0.18, min=-0.25, max=0.25, step=0.01, description="actual offset", continuous_update=False)
noise_amp_sl = widgets.FloatSlider(value=0.0, min=0.0, max=0.12, step=0.01, description="noise amp", continuous_update=False)
seed_sl = widgets.IntSlider(value=2, min=0, max=50, step=1, description="seed", continuous_update=False)
show_tube_cb = widgets.Checkbox(value=True, description="show tube envelope")
show_plans_cb = widgets.Checkbox(value=True, description="show planned paths")
showcase_btn = widgets.Button(description="Load Showcase", button_style="primary")
balanced_btn = widgets.Button(description="Load Balanced")

tube_out = widgets.Output()


def update_tube_view(_=None):
    tube_out.clear_output(wait=True)
    with tube_out:
        render_tube_mpc_dashboard(
            x1_init_sl.value,
            x2_init_sl.value,
            x1_ref_sl.value,
            horizon_sl.value,
            sim_steps_sl.value,
            design_bound_sl.value,
            actual_dist_sl.value,
            noise_amp_sl.value,
            seed_sl.value,
            show_tube_cb.value,
            show_plans_cb.value,
        )


def load_showcase(_=None):
    x1_init_sl.value = 0.0
    x2_init_sl.value = 0.0
    x1_ref_sl.value = -4.0
    horizon_sl.value = 5
    sim_steps_sl.value = 16
    design_bound_sl.value = 0.18
    actual_dist_sl.value = 0.18
    noise_amp_sl.value = 0.0
    seed_sl.value = 2
    show_tube_cb.value = True
    show_plans_cb.value = True


def load_balanced(_=None):
    x1_init_sl.value = 3.0
    x2_init_sl.value = -1.0
    x1_ref_sl.value = 3.5
    horizon_sl.value = 6
    sim_steps_sl.value = 14
    design_bound_sl.value = 0.10
    actual_dist_sl.value = 0.08
    noise_amp_sl.value = 0.01
    seed_sl.value = 1
    show_tube_cb.value = True
    show_plans_cb.value = True


showcase_btn.on_click(load_showcase)
balanced_btn.on_click(load_balanced)


for widget_item in [
    x1_init_sl,
    x2_init_sl,
    x1_ref_sl,
    horizon_sl,
    sim_steps_sl,
    design_bound_sl,
    actual_dist_sl,
    noise_amp_sl,
    seed_sl,
    show_tube_cb,
    show_plans_cb,
]:
    widget_item.observe(update_tube_view, names="value")

controls_row_1 = widgets.HBox([x1_init_sl, x2_init_sl, x1_ref_sl, horizon_sl])
controls_row_2 = widgets.HBox([sim_steps_sl, design_bound_sl, actual_dist_sl, noise_amp_sl, seed_sl])
controls_row_3 = widgets.HBox([show_tube_cb, show_plans_cb, showcase_btn, balanced_btn])

display(widgets.VBox([controls_row_1, controls_row_2, controls_row_3, tube_out]))
update_tube_view()
